# Fine-tuning CodeLlama 7B Instruct (V2) en Colab T4 con QLoRA

Este notebook entrena `codellama/CodeLlama-7b-Instruct-hf` con dataset V2 y reglas educativas estrictas.

## Pasos iniciales (obligatorio)
1. Men? **Entorno de ejecuci?n** -> **Cambiar tipo de entorno de ejecuci?n**.
2. Selecciona **GPU** (T4).
3. Ejecuta celdas en orden.

## 1) Setup GPU / info

In [ ]:
import os
import gc
import json
import random
from pathlib import Path

import torch

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))

## 2) Install dependencies
Si cambias versiones o aparece conflicto, reinicia entorno de ejecuci?n despu?s de instalar.

In [ ]:
!pip -q uninstall -y torch torchvision torchaudio transformers peft trl accelerate bitsandbytes triton
!pip -q install --index-url https://download.pytorch.org/whl/cu121 torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1
!pip -q install transformers==4.44.2 datasets==2.21.0 peft==0.12.0 trl==0.10.1 accelerate==0.33.0 bitsandbytes==0.43.3 triton==2.3.1 sentencepiece==0.1.99 protobuf==4.25.3

print("Instalaci?n completa. Si es la primera vez de esta sesi?n: Entorno de ejecuci?n -> Reiniciar entorno de ejecuci?n.")

## 3) Verificaci?n de entorno (CUDA + bitsandbytes)

In [ ]:
import torch
import bitsandbytes as bnb
import triton

print("torch:", torch.__version__, "cuda:", torch.version.cuda)
print("cuda_available:", torch.cuda.is_available())
print("bnb:", bnb.__version__)
print("triton:", triton.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA no est? disponible. Cambia a GPU T4 y reconecta.")

## 4) Login HuggingFace (opcional)
Necesario solo si el modelo requiere autenticaci?n en tu cuenta.

In [ ]:
from huggingface_hub import login

# login()  # descomenta si necesitas autenticarte
print("Login opcional listo.")

## 5) Cargar dataset V2 (Drive o subida manual)

In [ ]:
from google.colab import files

USE_DRIVE = True
DRIVE_TRAIN_PATH = "/content/drive/MyDrive/finetuning/train_v2.jsonl"
DRIVE_VAL_PATH = "/content/drive/MyDrive/finetuning/val_v2.jsonl"
LOCAL_DIR = Path("/content/data/finetuning")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    train_path = Path(DRIVE_TRAIN_PATH)
    val_path = Path(DRIVE_VAL_PATH)
else:
    print("Sube train_v2.jsonl y val_v2.jsonl")
    uploaded = files.upload()
    for fname, content in uploaded.items():
        (LOCAL_DIR / fname).write_bytes(content)
    train_path = LOCAL_DIR / "train_v2.jsonl"
    val_path = LOCAL_DIR / "val_v2.jsonl"

print("train_path:", train_path)
print("val_path:", val_path)
assert train_path.exists(), f"No existe {train_path}"
assert val_path.exists(), f"No existe {val_path}"

## 6) Preprocess: instruction/input/output -> text ([INST] + SYSTEM_TEXT)

In [ ]:
from datasets import load_dataset

SYSTEM_TEXT = 'Eres un asistente educativo de Python para ciencia de datos.\nREGLAS OBLIGATORIAS:\n1) Responde EXCLUSIVAMENTE en espa\u00f1ol.\n2) Prohibido usar rutas o leer archivos: /home, /kaggle, /content, ../input, pd.read_csv, read_parquet, read_excel.\n3) SIEMPRE incluye dataset sint\u00e9tico peque\u00f1o y el c\u00f3digo lo construye con pd.DataFrame.\n4) EXPLICACION espec\u00edfica, menciona columnas/variables reales.\nFORMATO EXACTO:\n## OBJETIVO\n## DATASET\n## CODIGO\n## EXPLICACION'\n
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
MAX_SEQ_LENGTH = 1024
SEED = 42

raw_ds = load_dataset(
    "json",
    data_files={"train": str(train_path), "validation": str(val_path)},
)


def format_example(example):
    instruction = (example.get("instruction") or "").strip()
    input_text = (example.get("input") or "").strip()
    output_text = (example.get("output") or "").strip()

    user_block = (
        f"Instruction:\n{instruction}\n\n"
        f"Input:\n{input_text}\n\n"
        "Output:\n"
    )

    prompt = (
        f"<s>[INST] <<SYS>>\n{SYSTEM_TEXT}\n<</SYS>>\n\n"
        f"{user_block} [/INST]\n"
        f"{output_text}</s>"
    )
    return {"text": prompt}


ds = raw_ds.map(format_example, remove_columns=raw_ds["train"].column_names)
print(ds)
print("Ejemplo text (inicio):")
print(ds["train"][0]["text"][:800])

## 7) Load model (4-bit NF4) + tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

model.config.use_cache = False
model.gradient_checkpointing_enable()
print("Modelo cargado en 4-bit.")

## 8) Config LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print(lora_config)

## 9) Train (SFTTrainer)
Ajustes recomendados para T4 16GB:
- `max_seq_length=1024` (si OOM baja a 768 o 512)
- `per_device_train_batch_size=1`
- `gradient_accumulation_steps=16` (batch efectivo 16)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

output_dir = "/content/models/codellama-edugen-v2"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,   # opcion: 3
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
    seed=SEED,
    report_to="none",
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"],
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

train_result = trainer.train()
train_result

## 10) Evaluate

In [ ]:
eval_metrics = trainer.evaluate()
print(eval_metrics)

## 11) Save artifacts
Guarda adapter + tokenizer + training args + estado del trainer.

In [ ]:
from pathlib import Path

save_dir = Path("/content/models/codellama-edugen-v2")
save_dir.mkdir(parents=True, exist_ok=True)

trainer.model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

with open(save_dir / "training_args.json", "w", encoding="utf-8") as f:
    json.dump(training_args.to_dict(), f, ensure_ascii=False, indent=2)

trainer.state.save_to_json(str(save_dir / "trainer_state.json"))

print("Guardado en:", save_dir)
print(sorted([p.name for p in save_dir.iterdir()]))

## 12) Quick generation tests (3 prompts)
Verifica formato y reglas: 4 secciones, sin rutas/read_csv/read_parquet/read_excel y en espa?ol.

In [ ]:
from transformers import pipeline

SYSTEM_TEXT = 'Eres un asistente educativo de Python para ciencia de datos.\nREGLAS OBLIGATORIAS:\n1) Responde EXCLUSIVAMENTE en espa\u00f1ol.\n2) Prohibido usar rutas o leer archivos: /home, /kaggle, /content, ../input, pd.read_csv, read_parquet, read_excel.\n3) SIEMPRE incluye dataset sint\u00e9tico peque\u00f1o y el c\u00f3digo lo construye con pd.DataFrame.\n4) EXPLICACION espec\u00edfica, menciona columnas/variables reales.\nFORMATO EXACTO:\n## OBJETIVO\n## DATASET\n## CODIGO\n## EXPLICACION'\n
gen = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    max_new_tokens=450,
    do_sample=True,
    temperature=0.2,
    top_p=0.9,
)

prompts = [
    "Tema: pandas_groupby, Nivel: principiante, Contexto: deportes, Tipo: tutorial",
    "Tema: matplotlib_basico, Nivel: intermedio, Contexto: ciencia, Tipo: desafio",
    "Tema: pandas_filtrado, Nivel: intermedio, Contexto: finanzas, Tipo: tutorial",
]

forbidden = ["/home", "/kaggle", "/content", "../input", "pd.read_csv", "read_parquet", "read_excel"]


def has_sections(text):
    required = ["## OBJETIVO", "## DATASET", "## CODIGO", "## EXPLICACION"]
    return all(r in text for r in required)


def has_forbidden(text):
    t = text.lower()
    return any(x.lower() in t for x in forbidden)


def looks_spanish(text):
    t = f" {text.lower()} "
    if any(ch in t for ch in "??????"):
        return True
    marks = [" el ", " la ", " de ", " y ", " para ", " datos ", " objetivo ", " explicacion "]
    return sum(1 for m in marks if m in t) >= 3

for p in prompts:
    full_prompt = (
        f"<s>[INST] <<SYS>>\n{SYSTEM_TEXT}\n<</SYS>>\n\n"
        f"Instruction:\nGenera un ejercicio educativo de Python.\n\n"
        f"Input:\n{p}\n\nOutput:\n [/INST]"
    )
    out = gen(full_prompt)[0]["generated_text"]
    generated = out.split("[/INST]", 1)[-1].strip()

    print("=" * 90)
    print("PROMPT:", p)
    print("- Tiene 4 secciones:", has_sections(generated))
    print("- Sin forbidden:", not has_forbidden(generated))
    print("- Espa?ol (heur?stico):", looks_spanish(generated))
    print("
GENERATED (preview):
")
    print(generated[:1600])

## 13) Export zip y descarga

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/models/codellama-edugen-v2.zip"
shutil.make_archive("/content/models/codellama-edugen-v2", "zip", "/content/models/codellama-edugen-v2")
print("Zip creado:", zip_path)
files.download(zip_path)

## 14) (Opcional) Merge LoRA + base model y guardado
Usa esta celda solo si tienes RAM/espacio suficiente.

In [ ]:
# Opcional: fusionar adapter con modelo base para inferencia directa
# Requiere memoria adicional y puede tardar.

# from peft import AutoPeftModelForCausalLM
# from transformers import AutoTokenizer
#
# peft_dir = "/content/models/codellama-edugen-v2"
# merged_dir = "/content/models/codellama-edugen-v2-merged"
#
# merged_model = AutoPeftModelForCausalLM.from_pretrained(
#     peft_dir,
#     torch_dtype=torch.float16,
#     low_cpu_mem_usage=True,
# )
# merged_model = merged_model.merge_and_unload()
#
# tok = AutoTokenizer.from_pretrained(peft_dir)
# merged_model.save_pretrained(merged_dir, safe_serialization=True)
# tok.save_pretrained(merged_dir)
# print("Modelo mergeado guardado en:", merged_dir)

## Notas de OOM (si falla memoria)
- Baja `MAX_SEQ_LENGTH` a 768 o 512.
- Mant?n `per_device_train_batch_size=1`.
- Sube `gradient_accumulation_steps` para conservar batch efectivo.
- Ejecuta limpieza:

```python
import gc, torch
gc.collect()
torch.cuda.empty_cache()
```